# 🧪 W4-D7 概念实验：第四周总复习（可执行版）

> 配套阅读：`第4周-Day7-第四周总复习.md`（知识全景图与 20 题测试在那边）
> 复习日安排：**公式自测 → 检索方法对决 → 分块甜点区 → W3↔W4 串联 → GraphRAG 多跳**
>
> 实验环境：纯 numpy + matplotlib。

## 实验 1：公式自测 —— 5 个 RAG 核心公式算出来对答案

余弦相似度、RRF、chunk 数、recall@k、IDF 单调性，每个都真的算一遍。

In [ ]:
import numpy as np
from collections import Counter

def check(no, desc, cond):
    print(f"  Q{no} {desc}: {'✓' if cond else '✗'}")

# Q1: 余弦相似度 —— 正交=0，同向=1
a, b = np.array([1.0, 0.0]), np.array([0.0, 3.0])
cos_orth = a @ b / (np.linalg.norm(a) * np.linalg.norm(b))
cos_same = a @ (2 * a) / (np.linalg.norm(a) * np.linalg.norm(2 * a))
check(1, f"正交={cos_orth:.1f} 同向={cos_same:.1f}", cos_orth == 0 and np.isclose(cos_same, 1))

# Q2: RRF —— 排名越靠前贡献越大
def rrf_score(rank, k=60): return 1 / (k + rank)
check(2, f"第1名({rrf_score(1):.4f}) > 第2名({rrf_score(2):.4f}) > 第10名({rrf_score(10):.4f})",
      rrf_score(1) > rrf_score(2) > rrf_score(10))

# Q3: chunk 数公式 —— 步长 = size - overlap
n_chunks = len(range(0, 1000, 120 - 40))          # 1000字, size=120, overlap=40
check(3, f"1000字/120块长/40重叠 = {n_chunks} 块", n_chunks == 13)

# Q4: recall@k 与 precision@k 的分母
retrieved = 5; relevant_total = 4; hit = 3
check(4, f"recall=3/4={3/4:.2f}, precision=3/5={3/5:.2f}",
      3 / relevant_total == 0.75 and hit / retrieved == 0.6)

# Q5: IDF 单调性 —— 出现在越多文档的词越不值钱
idf3, idf1 = np.log(1 + 10/3), np.log(1 + 10/1)   # 10篇语料
check(5, f"出现3篇({idf3:.2f}) < 出现1篇({idf1:.2f})", idf3 < idf1)

print("\n公式层 ✓ —— 这些是 Day2/Day3 的地基，也是面试高频")

## 实验 2：方法对决 —— BM25 / 向量 / RRF 混合（企业知识库场景）

换一个语料域（企业制度 FAQ，5 篇），跑 5 个真实问法的基准。
这组查询字面重叠充分——预期各路都能对；真正的分化在 Day3 的词汇鸿沟查询里。
混合检索的价值是**保险**：你无法预知用户问法落在字面侧还是语义侧。

In [ ]:
faq = [
    ("退款政策", "七天无理由退款。生鲜食品不支持退款。退款三个工作日到账。"),
    ("配送范围", "同城两小时送达。偏远地区次日达。满五十元免运费。"),
    ("会员制度", "充值五百元成为金卡会员。金卡享受九折优惠。积分可兑换商品。"),
    ("发票开具", "支持电子发票和纸质发票。发票在订单完成后申请。抬头需要准确填写。"),
    ("售后流程", "先联系在线客服。客服判断退货或换货。质检通过后完成售后。"),
]
names2 = [f[0] for f in faq]; texts2 = [f[0] + f[1] for f in faq]

def bigrams(s): return [s[i:i+2] for i in range(len(s) - 1)]

# --- BM25 ---
k1, b = 1.5, 0.75
tfs = [Counter(bigrams(t)) for t in texts2]
dfc = Counter()
for tf in tfs: dfc.update(set(tf))
avgdl = np.mean([sum(tf.values()) for tf in tfs])
idf2 = {t: np.log(1 + (5 - c + 0.5) / (c + 0.5)) for t, c in dfc.items()}

def bm25_scores(q):
    qt = bigrams(q)
    return np.array([sum(idf2.get(t, 0) * tf.get(t, 0) * (k1 + 1) /
                    (tf.get(t, 0) + k1 * (1 - b + b * sum(tf.values()) / avgdl)) for t in qt)
                     for tf in tfs])

# --- 向量（TF-IDF + L2）---
def tfidf_matrix(texts):
    tfl = [Counter(bigrams(t)) for t in texts]
    dfx = Counter()
    for tf in tfl: dfx.update(set(tf))
    vocab = sorted(dfx); vi = {t: i for i, t in enumerate(vocab)}
    M = np.zeros((len(texts), len(vocab)))
    for i, tf in enumerate(tfl):
        for t, f in tf.items():
            M[i, vi[t]] = f * np.log(1 + len(texts) / dfx[t])
    M /= np.linalg.norm(M, axis=1, keepdims=True) + 1e-9
    return M, vi
M2, vi2 = tfidf_matrix(texts2)

def vec_scores(q):
    tf = Counter(bigrams(q))
    qv = np.zeros(M2.shape[1])
    for t, f in tf.items():
        if t in vi2: qv[vi2[t]] = f
    qv /= np.linalg.norm(qv) + 1e-9
    return M2 @ qv

def rrf_hybrid(q, k=60):
    r1 = np.argsort(bm25_scores(q))[::-1]; r2 = np.argsort(vec_scores(q))[::-1]
    s = np.zeros(5)
    for r, i in enumerate(r1): s[i] += 1 / (k + r + 1)
    for r, i in enumerate(r2): s[i] += 1 / (k + r + 1)
    return s

bench = [
    ("生鲜能退吗", 0), ("多少钱免运费", 1), ("金卡打几折", 2),
    ("电子发票怎么开", 3), ("坏了怎么售后", 4),
]
print("5 个口语化查询，各方法 acc@1：")
for method, fn in [("BM25", bm25_scores), ("向量", vec_scores), ("RRF混合", rrf_hybrid)]:
    acc = sum(int(int(np.argmax(fn(q))) == gold) for q, gold in bench) / 5
    print(f"  acc@1({method}): {acc:.0%}")
print("\n→ 独立语料上复现 Day3 结论：混合检索对口语化查询最稳")

## 实验 3：分块甜点区 —— 覆盖率与精准率的乘积

块太小：关键事实被切断（覆盖率崩）；块太大：向量稀释 + 预算装不下几块（精准率降）。
两个因子分开模拟再相乘，U 型曲线就是"为什么要调 chunk_size"的全部理由。

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

rng = np.random.default_rng(7)
DOC, FACT, trials = 2000, 40, 400
starts = rng.uniform(0, DOC - FACT, trials)

sizes = np.array([40, 60, 80, 120, 160, 200, 280, 400])
coverage, precision = [], []
for c in sizes:
    step = int(c * 0.7)                       # overlap = 30%
    cs = np.arange(0, DOC, step)
    hit = np.zeros(trials, dtype=bool)
    for s0 in cs:
        hit |= (starts >= s0) & (starts + FACT <= s0 + c)
    coverage.append(hit.mean())
    # 精准率：块越多噪声块越多；块越大向量越稀释（金块均值随块长下降）
    n_chunks = len(cs)
    gold_s = rng.normal(0.86 - 0.30 * (c / 400), 0.05, trials)
    noise_max = rng.normal(0.62, 0.07, (trials, n_chunks)).max(axis=1)
    precision.append((gold_s > noise_max).mean())
coverage, precision = np.array(coverage), np.array(precision)
acc = coverage * precision

fig, ax = plt.subplots(figsize=(8.5, 4.4))
ax.plot(sizes, coverage, "s--", label="覆盖率：事实完整落进块内")
ax.plot(sizes, precision, "^--", label="精准率：金块排得进 top-k")
ax.plot(sizes, acc, "o-", lw=2, label="综合 = 覆盖 × 精准")
best = sizes[int(np.argmax(acc))]
ax.axvline(best, ls=":", color="gray"); ax.annotate(f"甜点区 ≈ {best} 字", xy=(best, acc.max()),
    xytext=(best + 60, acc.max() * 0.9), arrowprops=dict(arrowstyle="->"))
ax.set_xlabel("chunk_size（overlap=30%）"); ax.set_ylabel("概率")
ax.set_title("分块大小的 U 型权衡（模拟）")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 实验 4：W3 ↔ W4 串联 —— 检索做到 0.9 之后，瓶颈换了人

W3 学的是"把模型练好"，W4 学的是"把知识挂上去"。端到端答对率 =
检索命中 × 位置利用 × 生成忠实度。扫描检索召回从 0.5 到 0.98：
每个 +5pp 检索都线性传导 +4.1pp，但**乘法天花板 = 位置×忠实度 = 82.8%**——
检索逼近 0.95 后，剩下的缺口不在检索端，该去优化生成端了。

In [ ]:
pos_gain, fidelity = 0.92, 0.90          # 重排序置顶 + 引用约束后的固定项
recalls = np.arange(0.50, 0.99, 0.05)
e2e = recalls * pos_gain * fidelity

print(f"{'检索召回':>8}{'端到端答对率':>12}{'每+5pp检索的边际收益':>16}")
for i in range(1, len(recalls)):
    gain = e2e[i] - e2e[i-1]
    print(f"{recalls[i]:>8.2f}{e2e[i]:>12.1%}{gain*100:>13.1f}pp"
          + ("   ← 边际收益 < 3pp，瓶颈转移" if gain < 0.03 else ""))

print(f"\n召回 0.50 → 0.90：答对率 {e2e[0]:.0%} → {e2e[8]:.0%}（W4 的主战场）")
print(f"召回 0.90 → 0.95：答对率 {e2e[8]:.0%} → {e2e[9]:.0%}（继续卷检索性价比低）")
print("\n→ W3(训练决定 fidelity 上限) 与 W4(检索决定知识可达性) 在这里交汇：")
print("   优化资源按边际收益分配，而不是按技术爱好分配")

## 实验 5：GraphRAG lite —— 平面向量做不到的多跳

问题「和杨枝甘露共用原料的甜品有哪些」需要**两跳**：杨枝甘露 →(含)→ 芒果 ←(含)← 其他甜品。
平面向量检索一次只能召回"写得像查询的文本"；图谱把它变成两步邻居查询。
用邻接表 + BFS 实现多跳遍历。

In [ ]:
edges = [
    ("杨枝甘露", "含原料", "芒果"), ("杨枝甘露", "含原料", "西柚"),
    ("芒果班戟", "含原料", "芒果"), ("芒果西米捞", "含原料", "芒果"),
    ("草莓双皮奶", "含原料", "草莓"), ("姜撞奶", "含原料", "牛奶"),
    ("椰汁西米捞", "含原料", "椰浆"),
]

from collections import defaultdict, deque
out = defaultdict(list)
for s, r, o in edges:
    out[s].append((r, o))

def bfs_hops(start, hops):
    """从 start 出发走 hops 跳，返回 [(路径, 终点)]"""
    results, frontier = [], deque([(start, [start])])
    seen_depth = {start: 0}
    while frontier:
        node, path = frontier.popleft()
        d = len(path) - 1
        if d == hops:
            continue
        for r, nxt in out[node]:
            if nxt not in seen_depth or seen_depth[nxt] > d + 1:
                seen_depth[nxt] = d + 1
                frontier.append((nxt, path + [f"-{r}->", nxt]))
                if d + 1 <= hops:
                    results.append((path + [f"-{r}->", nxt], d + 1))
    return results

print("一跳：芒果甜品有哪些？")
for path, h in bfs_hops("芒果", 1):
    pass
one_hop = [e for e in edges if e[2] == "芒果"]
for s, r, o in one_hop:
    print(f"  芒果 ←含原料← {s}")

print("\n两跳：和杨枝甘露共用原料的甜品？")
mango_dishes = {s for s, r, o in edges if o == "芒果"}
shared = mango_dishes - {"杨枝甘露"}
print(f"  杨枝甘露 --含原料--> 芒果 <--含原料-- {shared}")
print("\n→ 平面 RAG：除非某段文本恰好写过这句关系，否则检索不到；")
print("  GraphRAG：把关系存成边，多跳查询 = 图遍历（代价：抽取与维护图谱的成本）")

## 结论

| 实验 | 对应 Day | 复习结论 |
|---|---|---|
| 公式自测 | Day2 | 余弦/RRF/IDF/recall 全部 ✓ |
| 方法对决 | Day3 | 独立语料复现：混合检索最稳 |
| 分块甜点区 | Day5 | 覆盖×精准的 U 型，甜点 ≈ 80-160 字 |
| W3↔W4 串联 | 全周 | 召回 0.9 后边际收益 <3pp，瓶颈转移 |
| GraphRAG | Day4 | 多跳问题需要图结构，平面向量做不到 |

→ 深入阅读：同目录 `.md` 版本（20 题综合测试 + 知识串联图）